## DECK & CARDS CLASSES

In [3]:
import random
class Card:
    def __init__(self, color, value):
        self.color = color
        self.value = value

class Deck:
    def __init__(self):
        # initialize the deck with 92 cards, 2 copies per color (8 total per number across all colors), 1 card of 0 for each color, 4 cards of Skip for each color
        self.cards = []
        for color in ['Red', 'Green', 'Blue', 'Yellow']:
            for number in range(1,10):
                for _ in range(2):    # 2 copies per color
                    strNum = str(number)
                    self.cards.append(Card(color,strNum))
            self.cards.append(Card(color,'0'))    # 1 0 for each color
            for i in range(4):
                self.cards.append(Card(color, "Skip")) # 4 cards of Skip for each color
        # shuffle the deck
        random.shuffle(self.cards)

    def draw_card(self):
        return self.cards.pop()

## NECESSARY GAME FUNCTIONS

In [4]:
def valid_move(card, top_card):
    if(card.value == top_card.value):
        return True
    if(card.color == top_card.color):
        return True
    return False

def get_valid_moves(hand, top_card):
    valid_moves = []
    for card in hand:
        x = valid_move(card, top_card)
        if(x == True):
            valid_moves.append(card)
    return valid_moves


def deal_cards(deck, num_players=3, cards_per_player=5):
    hands = []
    for i in range(num_players):
        hand = []  # this for each player, end par we will append to hands list
        for j in range(cards_per_player):
            hand.append(deck.draw_card())
        hands.append(hand)
    return hands

def initial_state():
    deck = Deck()                           # creates and shuffles 92 cards
    hands = deal_cards(deck)               # deal 5 cards to each player
    top_card = deck.cards.pop()            # flip top card to start

    # make sure top card is not a skip
    while(top_card.value == "Skip"):
        deck.cards.append(top_card)        # put skip back
        random.shuffle(deck.cards)
        top_card = deck.cards.pop()

    state = {
        'hands': hands,                    # [[p1 cards], [p2 cards], [p3 cards]]
        'top_card': top_card,
        'deck': deck.cards,                
        'current_player': 0,              # 0=player#1, 1=player#2, 2=player#3
            }
    return state

def apply_move(state, move):
    # Manual deepcopy becuz deepcopy nhi kaam karta
    new_hands = []
    for h in state['hands']:
        new_hands.append(h.copy())

    new_state = {
        'hands': new_hands,
        'top_card': state['top_card'],
        'deck': state['deck'].copy(),
        'current_player': state['current_player']
    }
    current = new_state['current_player']
    hand = new_state['hands'][current]  # get current player's hand
    top_card = new_state['top_card']
    deck = new_state['deck']
    valid_moves = get_valid_moves(hand,top_card)
    if(len(valid_moves) == 0): 
        #deck empty then pass turn
        if len(deck) > 0:
            hand.append(deck.pop())
        new_player = (current + 1) % 3
    else:
        if move is None:
            raise ValueError("move cannot be None when valid moves exist")
        hand.remove(move)
        new_state['top_card'] = move
        if(move.value == "Skip"):
            new_player = (current + 2) % 3
        else:
            new_player = (current + 1) % 3
    new_state['current_player'] = new_player
    return new_state


def evaluation(state, player_index, strategy, num_players=3):
    player_hand = state['hands'][player_index]
    COPP = 0
    CAI = len(player_hand)
    S = 0
    for opponent in range(num_players):
        if(opponent == player_index):   # skip player index
            continue 
        opponent_hand = state['hands'][opponent]
        COPP += len(opponent_hand)
    for card in player_hand:
        if(card.value == "Skip"):
            S += 1

    COPP = COPP / (num_players - 1)   # since COPP is average num cards in oppoenets so num_players - 1
    if (strategy == "defensive"):
        return 50 - 7*CAI + 2*COPP + 4*S
    elif(strategy == "offensive"):                                  
        return 50 - 5*CAI + 3*COPP + 2*S



## MINIMAX & EXPECTIMAX FUNCTIONS

In [5]:
def minimax(state, depth, is_maximizing, player_index):
    # base case
    if depth == 0 or is_terminal(state):
        return evaluation(state, player_index, "defensive")
    
    current = state['current_player']
    hand = state['hands'][current]
    top_card = state['top_card']
    valid_moves = get_valid_moves(hand, top_card)
    
    if len(valid_moves) == 0:
        moves = [None]          # only move is to draw
    else:
        moves = valid_moves

    if is_maximizing:           # P1's turn -> maximize score
        best_score = float('-inf')
        for move in moves:
            new_state = apply_move(state, move)
            score = minimax(new_state, depth-1, False, player_index)
            best_score = max(best_score, score)
        return best_score

    else:                       # opponent's turn -> minimize score
        best_score = float('inf')
        for move in moves:
            new_state = apply_move(state, move)
            score = minimax(new_state, depth-1, True, player_index)
            best_score = min(best_score, score)
        return best_score


def get_best_move_minimax(state, player_index, verbose=True):
    hand = state['hands'][player_index]
    top_card = state['top_card']
    valid_moves = get_valid_moves(hand, top_card)

    if len(valid_moves) == 0:
        depth2 = {"Draw": ["(no valid moves)"]}
        if verbose:
            print_turn_decision_tree(f"P{player_index+1} (Minimax)", [("Draw", 0.0)], 0, depth2_lines=depth2)
        return None             # must draw

    best_score = float('-inf')
    best_move = None

    scored_moves = []
    for move in valid_moves:
        new_state = apply_move(state, move)
        score = minimax(new_state, depth=3, is_maximizing=False, player_index=player_index)
        if verbose:
            print(f"  Play: {move.color} {move.value} -> Score: {score}")
        if score > best_score:
            best_score = score
            best_move = move
        scored_moves.append((move, float(score)))

    if best_move is not None:
        scored_actions = []
        depth2 = {}
        chosen_idx = 0
        for i, (m, val) in enumerate(scored_moves):
            action_text = f"Play {m.color} {m.value}"
            scored_actions.append((action_text, val))

            child_state = apply_move(state, m)
            lines, approx_val = explain_minimax(child_state, player_index, max_opp=4, max_next=4)
            depth2[action_text] = lines

            if m.color == best_move.color and m.value == best_move.value:
                chosen_idx = i
        if verbose:
            print_turn_decision_tree(f"P{player_index+1} (Minimax)", scored_actions, chosen_idx, depth2_lines=depth2)

    return best_move


def is_terminal(state):
    for hand in state['hands']:
        if len(hand) == 0:      # someone won
            return True
    if len(state['deck']) == 0: # deck empty
        return True
    return False


def expectimax(state, depth, player_index):
    # base case
    if depth == 0 or is_terminal(state):
        return evaluation(state, player_index, "offensive")
    
    current = state['current_player']
    hand = state['hands'][current]
    top_card = state['top_card']
    valid_moves = get_valid_moves(hand, top_card)

    if current == player_index:             # MAX node -> P2's turn
        if len(valid_moves) == 0:
            new_state = apply_move(state, None)
            return expectimax(new_state, depth-1, player_index)
        
        best_score = float('-inf')
        for move in valid_moves:
            new_state = apply_move(state, move)
            score = expectimax(new_state, depth-1, player_index)
            best_score = max(best_score, score)
        return best_score

    elif len(valid_moves) == 0:             # CHANCE node -> opponent must draw
        deck = state['deck']
        if len(deck) == 0:
            return evaluation(state, player_index, "offensive")
        
        total_score = 0
        for draw_card in deck:                          # consider each possible draw
            prob = 1 / len(deck)                        # equal probability for each card
            #manual copy becuz deepcopy nhi kaam karta
            new_hands = []
            for h in state['hands']:
                new_hands.append(h.copy())
            new_state = {
                'hands': new_hands,
                'top_card': state['top_card'],
                'deck': state['deck'].copy(),
                'current_player': state['current_player']
            }
            new_state['hands'][current].append(draw_card)
            new_state['deck'].remove(draw_card)
            new_state['current_player'] = (current + 1) % 3
            score = expectimax(new_state, depth-1, player_index)
            total_score += prob * score
        return total_score

    else:                                   # opponent turn -> random legal move
        scores = []
        for move in valid_moves:
            new_state = apply_move(state, move)
            score = expectimax(new_state, depth-1, player_index)
            scores.append(score)
        return sum(scores) / len(scores)    # average over all opponent moves


def get_best_move_expectimax(state, player_index, verbose=True):
    hand = state['hands'][player_index]
    top_card = state['top_card']
    valid_moves = get_valid_moves(hand, top_card)

    if len(valid_moves) == 0:
        depth2 = {"Draw": ["(no valid moves)"]}
        if verbose:
            print_turn_decision_tree(f"P{player_index+1} (Expectimax)", [("Draw", 0.0)], 0, depth2_lines=depth2)
        return None

    best_score = float('-inf')
    best_move = None

    scored_moves = []
    for move in valid_moves:
        new_state = apply_move(state, move)
        score = expectimax(new_state, depth=3, player_index=player_index)
        if verbose:
            print(f"  Play: {move.color} {move.value} -> Expected Score: {score:.2f}")
        if score > best_score:
            best_score = score
            best_move = move
        scored_moves.append((move, float(score)))

    if best_move is not None:
        scored_actions = []
        depth2 = {}
        chosen_idx = 0
        for i, (m, val) in enumerate(scored_moves):
            action_text = f"Play {m.color} {m.value}"
            scored_actions.append((action_text, val))

            # Deeper explanation (2 plies after root) for Expectimax
            child_state = apply_move(state, m)
            lines, approx_val = explain_expectimax(child_state, player_index, max_opp=4, max_next=4, max_draw=4)
            depth2[action_text] = lines

            if m.color == best_move.color and m.value == best_move.value:
                chosen_idx = i
        if verbose:
            print_turn_decision_tree(f"P{player_index+1} (Expectimax)", scored_actions, chosen_idx, depth2_lines=depth2)

    return best_move





def main():
    print("UNO game engine loaded.")
    print("Use `python uno_gui.py` to run the simulation GUI.")


## GAME TREE FUNCTIONS

In [ ]:
def print_turn_decision_tree(player_name, scored_actions, chosen_index, depth2_lines=None):
    print("\nDecision Tree (this turn):")
    print(player_name)

    if len(scored_actions) == 0:
        print("  (no actions)")
        return

    # Print as a simple root with children
    for i, (action_text, value) in enumerate(scored_actions):
        mark = "*" if i == chosen_index else " "
        print(f"  {mark} {action_text} -> {value:.2f}")

        if depth2_lines is not None and action_text in depth2_lines:
            for line in depth2_lines[action_text]:
                print(f"      {line}")


def explain_minimax(state_after_root_move, focus_player, max_opp=4, max_next=4):

    lines = []

    opp = state_after_root_move['current_player']
    opp_hand = state_after_root_move['hands'][opp]
    opp_top = state_after_root_move['top_card']
    opp_moves = get_valid_moves(opp_hand, opp_top)

    # If opponent cannot play, only option is Draw
    if len(opp_moves) == 0:
        opp_actions = [None]
    else:
        opp_actions = opp_moves

    shown_opp = opp_actions[:max_opp]
    option_values = []

    lines.append(f"Opponent P{opp+1} options (MIN):")

    for om in shown_opp:
        action_text = "Draw" if om is None else f"Play {om.color} {om.value}"
        reply_state = apply_move(state_after_root_move, om)

        nxt = reply_state['current_player']
        nxt_hand = reply_state['hands'][nxt]
        nxt_top = reply_state['top_card']
        nxt_moves = get_valid_moves(nxt_hand, nxt_top)

        # Next layer is MAX (because minimax toggles max/min each ply)
        if len(nxt_moves) == 0:
            nxt_actions = [None]
        else:
            nxt_actions = nxt_moves

        shown_nxt = nxt_actions[:max_next]
        leaf_scores = []

        lines.append(f"  - {action_text} -> Next P{nxt+1} options (MAX):")
        for nm in shown_nxt:
            nm_text = "Draw" if nm is None else f"Play {nm.color} {nm.value}"
            leaf_state = apply_move(reply_state, nm)
            leaf = evaluation(leaf_state, focus_player, "defensive")
            leaf_scores.append(leaf)
            lines.append(f"      {nm_text} -> leaf {leaf:.2f}")

        if len(nxt_actions) > max_next:
            lines.append(f"      ... {len(nxt_actions) - max_next} more moves not shown")

        max_pick = max(leaf_scores) if len(leaf_scores) > 0 else evaluation(reply_state, focus_player, "defensive")
        option_values.append(max_pick)
        lines.append(f"    MAX picks {max_pick:.2f}")

    if len(opp_actions) > max_opp:
        lines.append(f"  ... {len(opp_actions) - max_opp} more opponent moves not shown")

    min_pick = min(option_values) if len(option_values) > 0 else evaluation(state_after_root_move, focus_player, "defensive")
    lines.append(f"MIN picks {min_pick:.2f}")
    return lines, min_pick


def explain_expectimax(state_after_root_move, focus_player, max_opp=4, max_next=4, max_draw=4):

    lines = []

    opp = state_after_root_move['current_player']
    opp_hand = state_after_root_move['hands'][opp]
    opp_top = state_after_root_move['top_card']
    opp_moves = get_valid_moves(opp_hand, opp_top)

    # If opponent has no moves: CHANCE draw node
    if len(opp_moves) == 0:
        deck = state_after_root_move['deck']
        lines.append(f"Chance node: P{opp+1} must draw (deck size {len(deck)})")
        if len(deck) == 0:
            leaf = evaluation(state_after_root_move, focus_player, "offensive")
            lines.append(f"deck empty -> leaf {leaf:.2f}")
            return lines, leaf

        shown = deck[:max_draw]
        vals = []
        for dc in shown:
            # manual copy like expectimax()
            new_hands = []
            for h in state_after_root_move['hands']:
                new_hands.append(h.copy())
            new_state = {
                'hands': new_hands,
                'top_card': state_after_root_move['top_card'],
                'deck': state_after_root_move['deck'].copy(),
                'current_player': state_after_root_move['current_player']
            }
            new_state['hands'][opp].append(dc)
            new_state['deck'].remove(dc)
            new_state['current_player'] = (opp + 1) % 3

            leaf = evaluation(new_state, focus_player, "offensive")
            vals.append(leaf)
            lines.append(f"  - draw {dc.color} {dc.value} -> leaf {leaf:.2f}")

        if len(deck) > max_draw:
            lines.append(f"  ... {len(deck) - max_draw} more draws not shown")

        approx_e = sum(vals) / len(vals) if len(vals) > 0 else 0.0
        lines.append(f"Approx E = {approx_e:.2f}")
        return lines, approx_e

    lines.append(f"Opponent P{opp+1} options (RANDOM, averaged):")
    shown_opp = opp_moves[:max_opp]
    opp_vals = []

    for om in shown_opp:
        reply_state = apply_move(state_after_root_move, om)
        nxt = reply_state['current_player']
        nxt_hand = reply_state['hands'][nxt]
        nxt_top = reply_state['top_card']
        nxt_moves = get_valid_moves(nxt_hand, nxt_top)

        if len(nxt_moves) == 0:
            nxt_actions = [None]
        else:
            nxt_actions = nxt_moves

        shown_nxt = nxt_actions[:max_next]
        leaf_scores = []

        lines.append(f"  - Play {om.color} {om.value} -> Next P{nxt+1} options:")
        for nm in shown_nxt:
            nm_text = "Draw" if nm is None else f"Play {nm.color} {nm.value}"
            leaf_state = apply_move(reply_state, nm)
            leaf = evaluation(leaf_state, focus_player, "offensive")
            leaf_scores.append(leaf)
            lines.append(f"      {nm_text} -> leaf {leaf:.2f}")

        if len(nxt_actions) > max_next:
            lines.append(f"      ... {len(nxt_actions) - max_next} more moves not shown")

        # summarize by average of shown leaves.
        avg_leaf = sum(leaf_scores) / len(leaf_scores) if len(leaf_scores) > 0 else evaluation(reply_state, focus_player, "offensive")
        opp_vals.append(avg_leaf)
        lines.append(f"    AVG (shown) = {avg_leaf:.2f}")

    if len(opp_moves) > max_opp:
        lines.append(f"  ... {len(opp_moves) - max_opp} more opponent moves not shown")

    approx_avg = sum(opp_vals) / len(opp_vals) if len(opp_vals) > 0 else 0.0
    lines.append(f"Approx OPP average = {approx_avg:.2f}")
    return lines, approx_avg



def build_game_tree(state, depth, maximizing_player, algorithm="minimax"):

    for i, h in enumerate(state['hands']):
        if len(h) == 0:
            score = 1000 if i == maximizing_player else -1000
            return {'label': f'Win P{i+1}({score})', 'children': []}

    if len(state['deck']) == 0:
        strat = "defensive" if algorithm == "minimax" else "offensive"
        score = evaluation(state, maximizing_player, strat)
        return {'label': f'score={score:.0f}', 'children': []}

    if depth == 0:
        strat = "defensive" if algorithm == "minimax" else "offensive"
        score = evaluation(state, maximizing_player, strat)
        return {'label': f'score={score:.0f}', 'children': []}

    current = state['current_player']
    hand = state['hands'][current]
    top_card = state['top_card']
    valid_moves = get_valid_moves(hand, top_card)

    if algorithm == "minimax":
        is_max = (current == maximizing_player)
        node_label = f'P{current+1}(MAX)' if is_max else f'P{current+1}(MIN)'
    else:
        if current == maximizing_player:
            node_label = f'P{current+1}(MAX)'
        elif len(valid_moves) == 0:
            node_label = f'P{current+1}(CHANCE)'
        else:
            node_label = f'P{current+1}(RAND)'

    children = []

    if len(valid_moves) == 0:
        new_state = apply_move(state, None)
        child = build_game_tree(new_state, depth - 1, maximizing_player, algorithm)
        children.append({'label': 'Draw', 'children': [child]})
    else:
        for move in valid_moves:
            label = f'{move.color[0]}{move.value}'
            new_state = apply_move(state, move)
            child = build_game_tree(new_state, depth - 1, maximizing_player, algorithm)
            children.append({'label': label, 'children': [child]})

    return {'label': node_label, 'children': children}


def render_tree(node):
    label    = node['label']
    children = node['children']

    if not children:
        return [label], len(label), len(label) // 2

    GAP = 2
    child_renders = [render_tree(c) for c in children]
    child_widths  = [r[1] for r in child_renders]
    child_centers = [r[2] for r in child_renders]

    child_x = []
    x = 0
    for w in child_widths:
        child_x.append(x)
        x += w + GAP
    total_width  = x - GAP

    first_center = child_x[0]  + child_centers[0]
    last_center  = child_x[-1] + child_centers[-1]
    root_center  = (first_center + last_center) // 2
    label_start  = root_center - len(label) // 2

    if label_start + len(label) > total_width:
        total_width = label_start + len(label)
    if label_start < 0:
        shift        = -label_start
        child_x      = [cx + shift for cx in child_x]
        root_center  += shift
        label_start  += shift
        total_width  += shift

    branch_line = [' '] * (total_width + 2)
    for cx, cr in zip(child_x, child_centers):
        pos = cx + cr
        if   pos < root_center: branch_line[pos] = '/'
        elif pos > root_center: branch_line[pos] = '\\'
        else:                   branch_line[pos] = '|'

    label_line = (' ' * label_start + label).ljust(total_width)
    branch_str = ''.join(branch_line).rstrip()

    max_rows = max(len(r[0]) for r in child_renders)
    merged = []
    for row in range(max_rows):
        parts = []
        for r, cx in zip(child_renders, child_x):
            lines, w, _ = r
            seg = lines[row] if row < len(lines) else ' ' * w
            if parts:
                pad = cx - len(''.join(parts))
                parts.append(' ' * max(pad, 0))
            parts.append(seg.ljust(w))
        merged.append(''.join(parts))

    return [label_line, branch_str] + merged, total_width, root_center


def print_game_tree(state, player, algorithm="minimax", depth=2):
    algo_name = "Minimax" if algorithm == "minimax" else "Expectimax"
    hand = state['hands'][player]
    top  = state['top_card']
    print(f"\n{'='*60}")
    print(f"  GAME TREE  |  P{player+1} ({algo_name})  |  Top: {top.color} {top.value}")
    print(f"  Hand: {', '.join(c.color[0] + c.value for c in hand)}")
    print(f"  Depth: {depth}")
    print(f"{'='*60}\n")

    tree = build_game_tree(state, depth=depth, maximizing_player=player, algorithm=algorithm)
    lines, _, _ = render_tree(tree)
    for line in lines:
        print(line)
    print(f"\n{'='*60}")


def step_simulation(state, verbose=False, p3_mode="simulation", p3_manual_move=None):
    current = state['current_player']

    # Winner check
    for i, h in enumerate(state['hands']):
        if len(h) == 0:
            return state, f"P{i+1} already won.", True, i

    if current == 0:
        move = get_best_move_minimax(state, 0, verbose=verbose)
        actor = "P1"
    elif current == 1:
        move = get_best_move_expectimax(state, 1, verbose=verbose)
        actor = "P2"
    else:
        actor = "P3"
        if p3_mode == "manual":
            hand = state['hands'][2]
            top_card = state['top_card']
            valid = get_valid_moves(hand, top_card)
            if len(valid) == 0:
                move = None
            else:
                move = p3_manual_move
                if move is None:
                    raise ValueError("P3 manual mode needs p3_manual_move when a valid move exists")
        else:
            # Simulation mode: P3 uses minimax logic
            move = get_best_move_minimax(state, 2, verbose=verbose)

    if move is None:
        msg = f"{actor} draws"
    else:
        msg = f"{actor} plays {move.color} {move.value}"

    new_state = apply_move(state, move)

    # Winner check (after move)
    for i, h in enumerate(new_state['hands']):
        if len(h) == 0:
            return new_state, msg, True, i

    # Deck empty check
    if len(new_state['deck']) == 0:
        return new_state, "Deck empty. Game over.", True, None

    return new_state, msg, False, None


## INTERFACE TO RUN THE CODE

In [7]:
def pretty_card(card):
    return f"{card.color} {card.value}"

def print_state_simple(state, turn_no):
    print("\n" + "="*70)
    print(f"Turn {turn_no} | Current Player: P{state['current_player']+1}")
    print(f"Top Card: {pretty_card(state['top_card'])}")
    print(f"P1 hand ({len(state['hands'][0])}): " + ", ".join(pretty_card(c) for c in state['hands'][0]))
    print(f"P2 hand ({len(state['hands'][1])}): " + ", ".join(pretty_card(c) for c in state['hands'][1]))
    print(f"P3 hand ({len(state['hands'][2])}): " + ", ".join(pretty_card(c) for c in state['hands'][2]))
    print(f"Deck remaining: {len(state['deck'])}")

def choose_manual_p3_move(state):
    hand = state['hands'][2]
    top_card = state['top_card']
    valid = get_valid_moves(hand, top_card)

    if len(valid) == 0:
        print("P3 manual: no valid move, will draw.")
        return None

    print("\nP3 manual turn. Choose one card index to play:")
    for i, c in enumerate(valid):
        print(f"  [{i}] {c.color} {c.value}")

    while True:
        raw = input("Enter index: ").strip()
        if raw.isdigit():
            idx = int(raw)
            if 0 <= idx < len(valid):
                return valid[idx]
        print("Invalid index. Try again.")

mode = input("Choose mode for P3 ('manual' or 'simulation'): ").strip().lower()
if mode not in ("manual", "simulation"):
    print("Invalid choice. Defaulting to simulation.")
    mode = "simulation"

state = initial_state()
turn = 1
max_turns = 200

print_state_simple(state, turn_no=0)

while turn <= max_turns:
    cp = state['current_player']
    algo = "expectimax" if cp == 1 else "minimax"
    print_game_tree(state, player=cp, algorithm=algo, depth=2)

    if mode == "manual" and cp == 2:
        manual_move = choose_manual_p3_move(state)
    else:
        manual_move = None

    new_state, msg, game_over, winner = step_simulation(
        state,
        verbose=True,
        p3_mode=mode,
        p3_manual_move=manual_move,
    )
    print(f"\nAction: {msg}")
    state = new_state
    print_state_simple(state, turn_no=turn)

    if game_over:
        if winner is None:
            print("\nGame over: deck empty.")
        else:
            print(f"\nWinner: P{winner+1}")
        break

    turn += 1

if turn > max_turns:
    print("\nStopped at max_turns.")


Invalid choice. Defaulting to simulation.

Turn 0 | Current Player: P1
Top Card: Blue 6
P1 hand (5): Blue 4, Green 1, Blue 3, Blue 7, Blue 5
P2 hand (5): Red 2, Blue 8, Blue Skip, Red 7, Yellow 2
P3 hand (5): Red 1, Green 9, Blue 3, Blue 6, Red 9
Deck remaining: 76

  GAME TREE  |  P1 (Minimax)  |  Top: Blue 6
  Hand: B4, G1, B3, B7, B5
  Depth: 2

                                         P1(MAX)                                        
         /                   /                        \                        \
        B4                  B3                       B7                       B5        
         |                   |                        |                        |        
      P2(MIN)             P2(MIN)                  P2(MIN)                  P2(MIN)     
    /         \         /         \         /         |         \         /         \   
   B8       BSkip      B8       BSkip      B8       BSkip      R7        B8       BSkip 
    |         |         |         

## SIMULATION

In [8]:
import tkinter as tk

# ── Palette ────────────────────────────────────────────────────────────────────
BG        = "#0d0d1a"   # root window background
PANEL     = "#1a1a2e"   # inner panels / frames
CARD_BG   = "#12122a"   # hand canvas background
HEADER_BG = "#0f3460"   # active player header
ACCENT    = "#e94560"   # primary button / winner highlight
BTN_ALT   = "#533483"   # secondary buttons
BTN_P3SIM = "#1a6b3c"   # P3 simulation mode button
BTN_P3MAN = "#8b2500"   # P3 manual mode button
LOG_BG    = "#0a0a18"
LOG_FG    = "#c0c0e0"
TEXT_FG   = "#e0e0e0"

CARD_COLORS = {
    "Red":    "#c0392b",
    "Green":  "#27ae60",
    "Blue":   "#2471a3",
    "Yellow": "#d4ac0d",
}
CARD_TEXT = {
    "Red":    "#ffffff",
    "Green":  "#ffffff",
    "Blue":   "#ffffff",
    "Yellow": "#111111",
}


def _btn(parent, text, cmd, bg=BTN_ALT, fg="#ffffff", fs=10, px=14, py=6):
    return tk.Button(
        parent, text=text, command=cmd,
        bg=bg, fg=fg, activebackground=bg, activeforeground=fg,
        relief="flat", bd=0, padx=px, pady=py,
        font=("Segoe UI", fs, "bold"), cursor="hand2",
    )


class UnoSimGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("UNO — Adversarial Search Simulation")
        self.root.configure(bg=BG)

        self.state         = initial_state()
        self.game_over     = False
        self.p3_mode       = "simulation"   # "simulation" | "manual"
        self.manual_pending = False

        self._build_ui()
        self.render_all()
        self.refresh_scores()

    # ── Build UI ───────────────────────────────────────────────────────────────
    def _build_ui(self):
        r = self.root
        for row, wt in enumerate([0, 0, 1, 0, 0, 0, 1]):
            r.rowconfigure(row, weight=wt)
        r.columnconfigure(0, weight=1)

        # Title bar ─────────────────────────────────────────
        tbar = tk.Frame(r, bg=HEADER_BG, pady=10)
        tbar.grid(row=0, column=0, sticky="ew")
        tk.Label(tbar, text="♠  UNO — Adversarial Search Simulation",
                 bg=HEADER_BG, fg="#ffffff",
                 font=("Segoe UI", 14, "bold")).pack(side="left", padx=18)
        self.status_lbl = tk.Label(tbar, text="● Ready", bg=HEADER_BG, fg=ACCENT,
                                   font=("Segoe UI", 11, "bold"))
        self.status_lbl.pack(side="right", padx=18)

        # ── Card table ──────────────────────────────────────
        table = tk.Frame(r, bg=BG, padx=12, pady=10)
        table.grid(row=1, column=0, sticky="nsew")
        table.columnconfigure(0, weight=1)
        table.columnconfigure(1, weight=0)
        table.columnconfigure(2, weight=1)

        self.p2_canvas = self._hand_canvas(table, 440, 190)
        self.p2_canvas.grid(row=0, column=0, sticky="nsew", padx=(0, 10))

        # Centre: top card + deck count
        mid = tk.Frame(table, bg=PANEL, padx=14, pady=12)
        mid.grid(row=0, column=1, sticky="n")
        tk.Label(mid, text="TOP CARD", bg=PANEL, fg="#aaaacc",
                 font=("Segoe UI", 9, "bold")).pack()
        self.top_card_canvas = tk.Canvas(mid, width=100, height=150,
                                         bg=PANEL, highlightthickness=0)
        self.top_card_canvas.pack(pady=(4, 8))
        self.deck_lbl = tk.Label(mid, text="Deck: --", bg=PANEL, fg=TEXT_FG,
                                 font=("Segoe UI", 9))
        self.deck_lbl.pack()

        self.p3_canvas = self._hand_canvas(table, 440, 190)
        self.p3_canvas.grid(row=0, column=2, sticky="nsew", padx=(10, 0))

        # P1 below the table
        p1_wrap = tk.Frame(r, bg=BG, padx=12, pady=0)
        p1_wrap.grid(row=2, column=0, sticky="ew")
        p1_wrap.columnconfigure(0, weight=1)
        self.p1_canvas = self._hand_canvas(p1_wrap, 900, 190)
        self.p1_canvas.grid(row=0, column=0, sticky="ew")

        # ── Controls ────────────────────────────────────────
        ctrl = tk.Frame(r, bg=PANEL, padx=14, pady=10)
        ctrl.grid(row=3, column=0, sticky="ew")

        self.next_btn = _btn(ctrl, "▶  Next Turn", self.on_next_turn,
                             bg=ACCENT, fs=11, px=18, py=8)
        self.next_btn.pack(side="left", padx=(0, 8))

        _btn(ctrl, "↺  Reset", self.on_reset).pack(side="left", padx=(0, 20))

        # P3 mode toggle
        self.mode_btn = _btn(ctrl, "P3: Simulation Mode", self.toggle_p3_mode,
                             bg=BTN_P3SIM)
        self.mode_btn.pack(side="left")

        # ── Manual P3 picker row (hidden until P3 manual turn) ──
        self.manual_frame = tk.Frame(r, bg="#0f3460", padx=12, pady=8)
        # will be grid-placed dynamically
        tk.Label(self.manual_frame, text="P3 Manual — pick a card to play:",
                 bg="#0f3460", fg="#ffffff",
                 font=("Segoe UI", 10, "bold")).pack(side="left", padx=(0, 14))
        self.manual_cards_row = tk.Frame(self.manual_frame, bg="#0f3460")
        self.manual_cards_row.pack(side="left")

        # ── Divider ─────────────────────────────────────────
        tk.Frame(r, bg="#2a2a4a", height=1).grid(row=5, column=0, sticky="ew")

        # ── Log panels ──────────────────────────────────────
        logs = tk.Frame(r, bg=BG, padx=12, pady=10)
        logs.grid(row=6, column=0, sticky="nsew")
        logs.columnconfigure(0, weight=1)
        logs.columnconfigure(1, weight=1)
        logs.rowconfigure(0, weight=1)
        r.rowconfigure(6, weight=1)

        self.log        = self._log_panel(logs, "Action Log",                                col=0)
        self.scores_log = self._log_panel(logs, "Valid Moves & Scores (player about to act)", col=1)

    def _hand_canvas(self, parent, w, h):
        return tk.Canvas(parent, width=w, height=h, bg=CARD_BG,
                         highlightthickness=2, highlightbackground="#2a2a4a")

    def _log_panel(self, parent, title, col):
        lf = tk.LabelFrame(parent, text=f" {title} ", bg=BG, fg="#8888bb",
                           font=("Segoe UI", 9, "bold"), bd=1, relief="groove",
                           labelanchor="nw")
        lf.grid(row=0, column=col, sticky="nsew",
                padx=(0, 6) if col == 0 else (6, 0))
        t = tk.Text(lf, height=8, bg=LOG_BG, fg=LOG_FG,
                    font=("Consolas", 9), insertbackground=LOG_FG,
                    wrap="word", relief="flat", bd=4)
        t.pack(side="left", fill="both", expand=True)
        sb = tk.Scrollbar(lf, orient="vertical", command=t.yview, bg=PANEL,
                          troughcolor=BG)
        sb.pack(side="right", fill="y")
        t.configure(yscrollcommand=sb.set)
        return t

    # ── P3 mode toggle ─────────────────────────────────────
    def toggle_p3_mode(self):
        if self.p3_mode == "simulation":
            self.p3_mode = "manual"
            self.mode_btn.configure(text="P3: Manual Mode  ✎", bg=BTN_P3MAN)
        else:
            self.p3_mode = "simulation"
            self.mode_btn.configure(text="P3: Simulation Mode", bg=BTN_P3SIM)
            self._hide_manual_frame()

    # ── Next turn ──────────────────────────────────────────
    def on_next_turn(self):
        if self.game_over or self.manual_pending:
            return
        if self.state["current_player"] == 2 and self.p3_mode == "manual":
            self._show_manual_picker()
            return
        ns, msg, over, winner = step_simulation(self.state, verbose=False)
        self._apply(ns, msg, over, winner)

    def _show_manual_picker(self):
        self.manual_pending = True
        self.next_btn.configure(state="disabled")

        hand     = self.state["hands"][2]
        top_card = self.state["top_card"]
        valid    = get_valid_moves(hand, top_card)

        for w in self.manual_cards_row.winfo_children():
            w.destroy()

        if len(valid) == 0:
            _btn(self.manual_cards_row, "Draw  (forced)",
                 lambda: self._p3_pick(None),
                 bg="#7f1d1d", fg="#ffffff", py=5).pack(side="left", padx=4)
        else:
            for card in valid:
                c  = card
                bg = CARD_COLORS.get(c.color, "#555")
                fg = CARD_TEXT.get(c.color, "#fff")
                tk.Button(
                    self.manual_cards_row,
                    text=f"{c.color}\n{c.value}",
                    command=lambda mv=c: self._p3_pick(mv),
                    bg=bg, fg=fg, activebackground=bg, activeforeground=fg,
                    relief="flat", bd=0, padx=12, pady=5,
                    font=("Segoe UI", 9, "bold"), cursor="hand2", width=7,
                ).pack(side="left", padx=4)

        self.manual_frame.grid(row=4, column=0, sticky="ew")

    def _hide_manual_frame(self):
        self.manual_frame.grid_remove()
        self.manual_pending = False
        self.next_btn.configure(state="normal")

    def _p3_pick(self, card):
        self._hide_manual_frame()
        ns, msg, over, winner = step_simulation(
            self.state, verbose=False,
            p3_mode="manual", p3_manual_move=card,
        )
        self._apply(ns, msg, over, winner)

    def _apply(self, new_state, msg, over, winner):
        self.state = new_state
        self._write(self.log, msg)
        self.render_all()
        if over:
            self.game_over = True
            result = ("Deck empty — game over." if winner is None
                      else f"  P{winner+1} wins! 🎉")
            self.status_lbl.configure(text=result, fg=ACCENT)
            self._write(self.log, result)
            self.scores_log.delete("1.0", "end")
            self.scores_log.insert("end", "(game over — no more moves)\n")
            self._show_winner_card(winner)
        else:
            cp = new_state["current_player"]
            self.status_lbl.configure(
                text=f"● P{cp+1} to act", fg=ACCENT)
            self.refresh_scores()

    def _show_winner_card(self, winner):
        popup = tk.Toplevel(self.root)
        popup.overrideredirect(True)   # no title bar
        popup.configure(bg="#0d0d1a")

        # Size & center over the main window
        pw, ph = 420, 320
        self.root.update_idletasks()
        rx = self.root.winfo_rootx() + (self.root.winfo_width()  - pw) // 2
        ry = self.root.winfo_rooty() + (self.root.winfo_height() - ph) // 2
        popup.geometry(f"{pw}x{ph}+{rx}+{ry}")

        # Outer border frame
        border = tk.Frame(popup, bg=ACCENT, padx=3, pady=3)
        border.pack(fill="both", expand=True, padx=12, pady=12)

        inner = tk.Frame(border, bg="#1a1a2e")
        inner.pack(fill="both", expand=True)

        if winner is None:
            title_text = "GAME OVER"
            sub_text   = "The deck ran out."
            card_color = "#444466"
            algo_text  = ""
        else:
            algos = ["Minimax (defensive)", "Expectimax (offensive)", "Minimax (sim)"]
            player_colors = ["#2471a3", "#c0392b", "#27ae60"]
            title_text = f"P{winner+1}  WINS!"
            sub_text   = f"Algorithm: {algos[winner]}"
            card_color = player_colors[winner]
            algo_text  = ["defensive", "offensive", "simulation"][winner]

        # Trophy icon row
        tk.Label(inner, text="🏆", bg="#1a1a2e",
                 font=("Segoe UI Emoji", 36)).pack(pady=(22, 0))

        # Big winner title
        tk.Label(inner, text=title_text, bg="#1a1a2e", fg=card_color,
                 font=("Segoe UI", 28, "bold")).pack(pady=(6, 2))

        # Sub-text
        tk.Label(inner, text=sub_text, bg="#1a1a2e", fg="#aaaacc",
                 font=("Segoe UI", 11)).pack(pady=(0, 18))

        # Close button
        def close_and_reset():
            popup.destroy()
            self.on_reset()

        btn_row = tk.Frame(inner, bg="#1a1a2e")
        btn_row.pack()
        _btn(btn_row, "Play Again", close_and_reset,
             bg=ACCENT, fs=11, px=20, py=8).pack(side="left", padx=8)
        _btn(btn_row, "Close", popup.destroy,
             bg="#333355", fs=11, px=20, py=8).pack(side="left", padx=8)

    # ── Reset ──────────────────────────────────────────────
    def on_reset(self):
        self.state = initial_state()
        self.game_over = False
        self._hide_manual_frame()
        self.status_lbl.configure(text="● New game", fg=ACCENT)
        self.log.delete("1.0", "end")
        self.scores_log.delete("1.0", "end")
        self.render_all()
        self.refresh_scores()

    # ── Logs ───────────────────────────────────────────────
    def _write(self, widget, text):
        widget.insert("end", text + "\n")
        widget.see("end")

    def refresh_scores(self):
        self.scores_log.delete("1.0", "end")
        if self.game_over:
            return
        cp   = self.state["current_player"]
        algo = "Minimax" if cp in (0, 2) else "Expectimax"
        self.scores_log.insert("end", f"P{cp+1} ({algo})\n")
        try:
            pairs = scored_valid_moves(self.state, cp)
        except Exception as exc:
            self.scores_log.insert("end", f"  error: {exc}\n")
            return
        for label, score in pairs:
            self.scores_log.insert("end", f"  {label}: {score:.2f}\n")
        self.scores_log.see("end")

    # ── Card rendering ─────────────────────────────────────
    def _draw_card(self, canvas, x, y, w, h, card, glow=False):
        bg  = CARD_COLORS.get(card.color, "#555555")
        fg  = CARD_TEXT.get(card.color, "#ffffff")
        rim = ACCENT if glow else "#000000"
        bw  = 3 if glow else 1

        # Drop shadow
        canvas.create_rectangle(x+4, y+4, x+w+4, y+h+4,
                                 fill="#1a1a1a", outline="")
        # Card body
        canvas.create_rectangle(x, y, x+w, y+h,
                                 fill=bg, outline=rim, width=bw)
        # Decorative inner oval
        canvas.create_oval(x+8, y+12, x+w-8, y+h-12,
                            fill="#2a2a5a", outline="#3a3a6a", width=1)
        # Value (large)
        canvas.create_text(x+w//2, y+h//2 - 8,
                            text=str(card.value), fill=fg,
                            font=("Segoe UI", 16, "bold"))
        # Colour label (small, bottom)
        canvas.create_text(x+w//2, y+h-14,
                            text=card.color, fill=fg,
                            font=("Segoe UI", 7, "bold"))

    def render_hand(self, canvas, title, cards, active=False):
        canvas.delete("all")
        # Header strip
        hdr_bg = "#1a3a6a" if active else "#181830"
        rim    = ACCENT   if active else "#2a2a4a"
        canvas.configure(highlightbackground=rim)
        canvas.create_rectangle(0, 0, 5000, 30, fill=hdr_bg, outline="")
        canvas.create_text(12, 15, anchor="w",
                            text=f"{title}  ({len(cards)} cards)",
                            fill="#ffffff", font=("Segoe UI", 10, "bold"))

        cw, ch = 62, 94
        pad    = 8
        x0, y0 = 10, 38
        canvas_w = int(canvas.winfo_width()) or 440
        per_row  = max(1, (canvas_w - x0 - 10) // (cw + pad))

        for i, c in enumerate(cards):
            col_i = i % per_row
            row_i = i // per_row
            self._draw_card(canvas,
                            x0 + col_i * (cw + pad),
                            y0 + row_i * (ch + pad),
                            cw, ch, c)

    def render_top_card(self):
        self.top_card_canvas.delete("all")
        self._draw_card(self.top_card_canvas, 0, 0, 100, 150,
                        self.state["top_card"], glow=True)
        self.deck_lbl.configure(text=f"Deck: {len(self.state['deck'])}")

    def render_all(self):
        cp = self.state["current_player"] if not self.game_over else -1
        self.render_top_card()
        self.render_hand(self.p1_canvas,
                         "P1  Minimax  (defensive)",
                         self.state["hands"][0], active=(cp == 0))
        self.render_hand(self.p2_canvas,
                         "P2  Expectimax  (offensive)",
                         self.state["hands"][1], active=(cp == 1))
        p3_label = ("P3  Minimax  — MANUAL"
                    if self.p3_mode == "manual"
                    else "P3  Minimax  (simulation)")
        self.render_hand(self.p3_canvas,
                         p3_label,
                         self.state["hands"][2], active=(cp == 2))


# ── scored_valid_moves (used by refresh_scores) ────────────────────────────────
def scored_valid_moves(state, player_index):
    hand       = state["hands"][player_index]
    top_card   = state["top_card"]
    valid_moves = get_valid_moves(hand, top_card)

    if len(valid_moves) == 0:
        ns = apply_move(state, None)
        if player_index == 1:
            s = expectimax(ns, depth=3, player_index=player_index)
        else:
            s = minimax(ns, depth=3, is_maximizing=False,
                        player_index=player_index)
        return [("Draw", float(s))]

    out = []
    if player_index == 1:
        for mv in valid_moves:
            ns = apply_move(state, mv)
            s  = expectimax(ns, depth=3, player_index=player_index)
            out.append((f"Play {mv.color} {mv.value}", float(s)))
    else:
        for mv in valid_moves:
            ns = apply_move(state, mv)
            s  = minimax(ns, depth=3, is_maximizing=False,
                         player_index=player_index)
            out.append((f"Play {mv.color} {mv.value}", float(s)))
    return out


# ── Launch ─────────────────────────────────────────────────────────────────────
root = tk.Tk()
app  = UnoSimGUI(root)
root.minsize(1020, 860)
root.mainloop()


>>> Minimax tree for P1:

  GAME TREE  |  P1 (Minimax)  |  Top: Blue 4
  Hand: Y3, R5, B2, R5, Y9
  Depth: 2

 P1(MAX)
    |
   B2   
    |   
 P2(MIN)
    |   
   B7   
    |   
score=31



>>> Expectimax tree for P2:

  GAME TREE  |  P2 (Expectimax)  |  Top: Blue 4
  Hand: B7, RSkip, G7, G9, R4
  Depth: 2

P1(RAND)
    |
   B2   
    |   
 P2(MAX)
    |   
   B7   
    |   
score=46



## Comparison of Both Algorithms

### Minimax & Expectimax
UNO uses two algorithms for AI:

- **Minimax:** makes assumption that the opponent will make move that will be worst for us (in context of this assignment, it makes the assumption that the opponent will make a move that will minimize your score).
- **Expectimax:** assumes uncertainty/chance and averages outcomes, so it favors moves with high expected value instead of only worst case safety.

**NOTE** ***Both the algorithms will choose the move which has highest score, its just that they have different way to evaluate the scores.***
### Evaluation Function
The evaluation is based on:

- **CAI:** cards in current AI player's hand
- **COPP:** average cards in opponents' hands
- **S:** number of Skip cards in hand

Form used in code:

- **Defensive (Minimax):** `50 - 7*CAI + 2*COPP + 4*S`
- **Offensive (Expectimax):** `50 - 5*CAI + 3*COPP + 2*S`

Interpretation:

- For Expectimax, I increased `COPP` and `S` coefficients to maximize the score. Intuitively, if we think about it, it means if the average number of cards of other players is higher, I will play more aggressively because I have the upper hand, and if I have a higher number of "Skip" cards, I will use them more frequently.
- For mimimax, I increased the coefficient of `CAI` and decreased the coefficient of `COPP` to play defensively. If the average cards of the other players are higher, I would still play it safer, unlike in Expectimax.

### Which Algorithm Performed Best
- **Expectimax:** tends to perform strongly when uncertainty creates opportunities for high expected value.
- **Minimax:** more consistent, especially for games like this where you want to choose the best move for you and the worst move for the other player.

So, superiority is: **Expectimax is often better for aggressive and uncertain situations, while Minimax is often better for worst case safety.**